# Caltrans 里程桩 vs PeMS 站点可视化对比

In [1]:
import pandas as pd
import numpy as np
import json
import os
import glob
import folium
from tqdm import tqdm

# ============== 配置 ==============

# Caltrans GeoJSON 文件路径
CALTRANS_GEOJSON = "./SHN_Postmiles_Tenth.geojson"

# PeMS 元数据目录
META_DIR = "../d03_meta"

# 目标高速
TARGET_ROUTE = 99

# 输出目录
OUTPUT_DIR = "./output/visualization"
os.makedirs(OUTPUT_DIR, exist_ok=True)

## 1. 加载 Caltrans 里程桩数据

In [2]:
print(f"加载 Caltrans 数据: {CALTRANS_GEOJSON}")
print("文件较大，请稍候...")

with open(CALTRANS_GEOJSON, 'r') as f:
    data = json.load(f)

print(f"总特征数: {len(data['features'])}")

# 只提取目标高速
# 只提取目标高速
records = []
for feature in tqdm(data['features'], desc="筛选数据"):
    props = feature.get('properties', {})
    if props.get('Route') != TARGET_ROUTE:
        continue
    
    geom = feature.get('geometry', {})
    coords = geom.get('coordinates', [None, None])
    
    records.append({
        'Route': props.get('Route'),
        'County': props.get('County'),
        'District': props.get('District'),
        'PM': props.get('PM'),           # 改为 PM
        'AlignCode': props.get('AlignCode'),
        'Direction': props.get('Direction'),  # 新增 Direction 字段
        'Odometer': props.get('Odometer'),
        'Longitude': coords[0],
        'Latitude': coords[1],
    })

caltrans_df = pd.DataFrame(records)
print(f"\nRoute {TARGET_ROUTE} 里程桩数: {len(caltrans_df)}")
print(f"\nAlignCode 分布:")
print(caltrans_df['AlignCode'].value_counts())

加载 Caltrans 数据: ./SHN_Postmiles_Tenth.geojson
文件较大，请稍候...


KeyboardInterrupt: 

## 2. 加载 PeMS 站点数据

In [ ]:
META_COLUMNS = [
    'ID', 'Fwy', 'Dir', 'District', 'County', 'City',
    'State_PM', 'Abs_PM', 'Latitude', 'Longitude', 'Length',
    'Type', 'Lanes', 'Name', 'User_ID_1', 'User_ID_2',
    'User_ID_3', 'User_ID_4'
]

# 加载 D3 元数据（99号高速主要在 D3, D6, D10）
all_meta = []
for district in ['d03', 'd06', 'd10']:
    pattern = os.path.join(META_DIR, f"{district}_text_meta_*.txt")
    files = glob.glob(pattern)
    if files:
        meta_file = sorted(files)[-1]
        df = pd.read_csv(meta_file, sep='\t', names=META_COLUMNS, header=0, 
                         dtype={'ID': str, 'Fwy': str})
        all_meta.append(df)
        print(f"{district}: {len(df)} 条")

pems_df = pd.concat(all_meta, ignore_index=True)

# 筛选目标高速
pems_target = pems_df[pems_df['Fwy'] == str(TARGET_ROUTE)].copy()
pems_target = pems_target[pems_target['Latitude'].notna()]

print(f"\nPeMS {TARGET_ROUTE} 号高速站点: {len(pems_target)}")
print(f"\n方向分布:")
print(pems_target['Dir'].value_counts())
print(f"\n类型分布:")
print(pems_target['Type'].value_counts())

d03: 1903 条

PeMS 99 号高速站点: 334

方向分布:
Dir
S    175
N    159
Name: count, dtype: int64

类型分布:
Type
ML    160
OR     70
HV     55
FR     46
FF      3
Name: count, dtype: int64


## 3. 可视化对比

In [ ]:

# 按方向分离 Caltrans 数据
caltrans_right = caltrans_df[
    (caltrans_df['AlignCode'].isin(['Right', 'Right Side']))
].sort_values('Odometer')

caltrans_left = caltrans_df[
    (caltrans_df['AlignCode'].isin(['Left', 'Left Side']))
].sort_values('Odometer')

print(f"Caltrans Right (N/E): {len(caltrans_right)} 个点")
print(f"Caltrans Left (S/W): {len(caltrans_left)} 个点")

# 按方向分离 PeMS 数据
pems_n = pems_target[pems_target['Dir'] == 'N'].sort_values('Abs_PM')
pems_s = pems_target[pems_target['Dir'] == 'S'].sort_values('Abs_PM')

print(f"\nPeMS N: {len(pems_n)} 个站点")
print(f"PeMS S: {len(pems_s)} 个站点")

Caltrans Right (N/E): 4270 个点
Caltrans Left (S/W): 4270 个点

PeMS N: 159 个站点
PeMS S: 175 个站点


In [ ]:
# 创建地图
center_lat = caltrans_df['Latitude'].mean()
center_lon = caltrans_df['Longitude'].mean()

m = folium.Map(location=[center_lat, center_lon], zoom_start=8, tiles=None)

# 添加底图
folium.TileLayer('OpenStreetMap', name='OSM').add_to(m)
folium.TileLayer(
    tiles='https://mt1.google.com/vt/lyrs=m&x={x}&y={y}&z={z}',
    attr='Google', name='Google 街道'
).add_to(m)
folium.TileLayer(
    tiles='https://mt1.google.com/vt/lyrs=y&x={x}&y={y}&z={z}',
    attr='Google', name='Google 混合'
).add_to(m)

# ========== Caltrans 里程点 ==========
# Right 方向（绿色点）
caltrans_right_group = folium.FeatureGroup(name='Caltrans Right (N/E)')
for _, row in caltrans_right.iterrows():
    folium.CircleMarker(
        [row['Latitude'], row['Longitude']],
        radius=3,
        color='#4CAF50',
        fill=True,
        fillColor='#4CAF50',
        fillOpacity=0.8,
        weight=1,
        popup=f"Abs_PM={row['Odometer']:.2f}<br>County={row['County']}",
        tooltip=f"Abs_PM={row['Odometer']:.1f}"
    ).add_to(caltrans_right_group)
caltrans_right_group.add_to(m)

# Left 方向（青色点）
caltrans_left_group = folium.FeatureGroup(name='Caltrans Left (S/W)')
for _, row in caltrans_left.iterrows():
    folium.CircleMarker(
        [row['Latitude'], row['Longitude']],
        radius=3,
        color='#00BCD4',
        fill=True,
        fillColor='#00BCD4',
        fillOpacity=0.8,
        weight=1,
        popup=f"Abs_PM={row['Odometer']:.2f}<br>County={row['County']}",
        tooltip=f"Abs_PM={row['Odometer']:.1f}"
    ).add_to(caltrans_left_group)
caltrans_left_group.add_to(m)

# ========== PeMS 站点 ==========
station_colors = {
    'ML': '#E53935',  # 红
    'HV': '#8E24AA',  # 紫
    'OR': '#FF9800',  # 橙
    'FR': '#795548',  # 棕
    'FF': '#FFC107',  # 黄
}

# PeMS N 方向
pems_n_group = folium.FeatureGroup(name='PeMS 99N')
for _, row in pems_n.iterrows():
    color = station_colors.get(row['Type'], '#888')
    folium.CircleMarker(
        [row['Latitude'], row['Longitude']],
        radius=7,
        color='white',
        weight=2,
        fill=True,
        fillColor=color,
        fillOpacity=0.9,
        popup=f"{row['ID']}<br>{row['Type']}<br>PM={row['Abs_PM']:.2f}",
        tooltip=f"{row['ID']} ({row['Type']})"
    ).add_to(pems_n_group)
pems_n_group.add_to(m)

# PeMS S 方向
pems_s_group = folium.FeatureGroup(name='PeMS 99S')
for _, row in pems_s.iterrows():
    color = station_colors.get(row['Type'], '#888')
    folium.CircleMarker(
        [row['Latitude'], row['Longitude']],
        radius=7,
        color='black',
        weight=2,
        fill=True,
        fillColor=color,
        fillOpacity=0.9,
        popup=f"{row['ID']}<br>{row['Type']}<br>PM={row['Abs_PM']:.2f}",
        tooltip=f"{row['ID']} ({row['Type']})"
    ).add_to(pems_s_group)
pems_s_group.add_to(m)

# 图例
legend = """
<div style="position:fixed; bottom:50px; left:50px; z-index:1000;
            background:white; padding:10px; border:2px solid #333; border-radius:5px;">
    <div style="font-weight:bold; margin-bottom:8px;">图例</div>
    <div style="font-weight:bold; margin-top:5px;">Caltrans 里程点:</div>
    <div><span style="color:#4CAF50;">●</span> Right (N/E方向)</div>
    <div><span style="color:#00BCD4;">●</span> Left (S/W方向)</div>
    <div style="font-weight:bold; margin-top:8px;">PeMS 站点:</div>
    <div>○ 白边=N方向, 黑边=S方向</div>
    <div><span style="color:#E53935;">●</span> ML (主线)</div>
    <div><span style="color:#8E24AA;">●</span> HV (HOV)</div>
    <div><span style="color:#FF9800;">●</span> OR (入口)</div>
    <div><span style="color:#795548;">●</span> FR (出口)</div>
</div>
"""
m.get_root().html.add_child(folium.Element(legend))

folium.LayerControl().add_to(m)

# 保存
output_path = os.path.join(OUTPUT_DIR, f'caltrans_vs_pems_{TARGET_ROUTE}.html')
m.save(output_path)
print(f"地图已保存: {output_path}")

m

## 4. 观察要点

在地图上检查:

1. **Caltrans 里程线是否与底图道路重合？**
   - 绿色线 (Right) 应该在 N/E 方向车道上
   - 青色线 (Left) 应该在 S/W 方向车道上

2. **PeMS 站点与 Caltrans 里程线的关系？**
   - 白边站点 (N方向) 应该靠近绿色线
   - 黑边站点 (S方向) 应该靠近青色线

3. **偏移情况？**
   - 站点是否落在正确的车道上
   - 是否有站点落在反向车道或路外